# How to run simulations in parallel

For a simulator that takes one second per call, 10,000 simulations take almost three hours on one CPU. `sbi` can spread the simulations over several processes with `joblib`: pass `num_workers`.

| You have | Use |
|---|---|
| a prior (or proposal) and want training data | `simulate_for_sbi(simulator, prior, num_simulations, num_workers=8)` |
| parameters already, e.g. from a design or a posterior | `simulate_from_theta(simulator, theta, num_workers=8)` |

Both run the same code. They differ in one default: `simulate_for_sbi` expects a simulator that takes a batch of parameter sets per call, `simulate_from_theta` one that takes a single parameter set. Set `simulator_is_batched` to say which one yours is.

Fast or vectorized simulators, such as most synthetic ones, rarely gain from parallel workers.

In [1]:
import time

import numpy as np
import torch

from sbi.utils import BoxUniform


def simulator(theta):
    """A slow simulator: one parameter set in, two summary statistics out."""
    time.sleep(0.05)  # Stands in for an expensive computation.
    theta = np.asarray(theta)
    trace = theta[0] + theta[1] * np.random.randn(100)
    return np.array([trace.mean(), trace.std()])


prior = BoxUniform(low=torch.zeros(2), high=torch.ones(2))

## Training data from a prior

`simulate_for_sbi` samples parameters from the prior and simulates them. Our simulator takes one parameter set per call, so we pass `simulator_is_batched=False`.

In [2]:
from sbi.inference import simulate_for_sbi

start = time.perf_counter()
theta, x = simulate_for_sbi(
    simulator,
    prior,
    num_simulations=200,
    num_workers=4,
    simulator_is_batched=False,
    seed=0,
    show_progress_bar=False,
)
print(
    f"{time.perf_counter() - start:.1f} s for {len(x)} simulations; x has shape {tuple(x.shape)}"
)

4.3 s for 200 simulations; x has shape (200, 2)


One worker would need about 10 s here (200 × 0.05 s). The first parallel call also takes a few seconds to start the workers; later calls reuse them.

The simulator receives the parameters as a tensor: on the CPU if there is more than one worker, otherwise on the device of the prior. A simulator that uses numpy converts them, as ours does with `np.asarray` (for a prior on a GPU: `np.asarray(theta.cpu())`). The returned `theta` and `x` are float32 tensors, ready for `append_simulations`.

## Parameters you already have

`simulate_from_theta` simulates given parameters and returns the outputs in the same order. The simulator receives the parameters as the type of `theta`, a tensor or a numpy array. Tensor outputs of one shape come back as one tensor, arrays and numbers as one numpy array, and anything else as a list.

In [3]:
from sbi.utils import simulate_from_theta

theta = prior.sample((200,))
x = simulate_from_theta(simulator, theta, num_workers=4, show_progress_bar=False)
print(type(x).__name__, x.shape)

ndarray (200, 2)


Our simulator returns numpy arrays, so `x` is a numpy array. For `append_simulations`, convert it with `torch.as_tensor(x, dtype=torch.float32)`.

## Batched simulators

If your simulator takes a batch of parameter sets per call, pass `simulator_is_batched=True` (the default in `simulate_for_sbi`). `simulation_batch_size` then sets the number of parameter sets per call. The default is 1; a vectorized simulator is faster with larger batches, and `None` passes all parameter sets in one call.

A simulator that runs on a GPU is usually best run batched on one worker. With more workers, every worker process creates its own CUDA context, which takes several hundred MB of GPU memory, and the workers share one GPU.

If a batched simulator does not return one output per parameter set, or if it raises an exception, the error suggests `simulator_is_batched=False`. The opposite mistake gets no such hint: a batched simulator passed with `simulator_is_batched=False` receives one parameter set of shape `(num_parameters,)` per call, so code such as `theta[:, 0]` raises an `IndexError`, and code that sizes its outputs by `len(theta)` runs, but each output gains a dimension.

## Simulators that write files

Some simulators run an external program that writes its results to disk. Return the file path: `simulate_from_theta` returns outputs that are not numbers as a list, in the order of `theta`. `simulate_for_sbi` needs numeric outputs, so load and summarize the files first.

In [4]:
import uuid


def run_external_program(theta):
    # Names built from theta can collide, also between workers.
    path = f"simulation_{uuid.uuid4().hex}.npy"
    # Your program would write its result to `path` here.
    return path


paths = simulate_from_theta(
    run_external_program, prior.sample((3,)), num_workers=2, show_progress_bar=False
)
print(paths)

['simulation_567f8454e760427e883b5151e3e7ee83.npy', 'simulation_06c34924234641069fd6a294c5bcbb57.npy', 'simulation_df8cb944cd8449d0bd15b8f9623b9281.npy']


## Failed simulations

An exception in the simulator stops all simulations and is raised, with a note that names the failed parameter set. In a long run, you may want to keep the finished simulations instead. Catch the exception in your simulator and return NaN in the shape of its output:

In [5]:
def fragile_simulator(theta):
    if theta[0] > 0.9:
        raise RuntimeError("The solver diverged.")
    return simulator(theta)


def tolerant_simulator(theta):
    try:
        return fragile_simulator(theta)
    except RuntimeError:
        return np.full(2, np.nan)


x = simulate_from_theta(
    tolerant_simulator, theta, num_workers=4, show_progress_bar=False
)
print(f"{np.isnan(x).any(axis=1).sum()} of {len(x)} simulations failed")

22 of 200 simulations failed


Failures often cluster in one region of the parameter space, as here for `theta[0] > 0.9`. If you discard the failed simulations, the posterior describes only the parameters that do not fail. See the FAQ entry [Can the algorithms deal with invalid data, e.g., NaN or inf?](../faq/question_02_nans.md) for how `sbi` handles NaN in the training data.

## Reproducible simulations

Pass a `seed`. Every simulator call then gets its own seed, derived from `seed`, so the outputs are the same for any `num_workers`. This holds for noise drawn on the CPU. PyTorch's GPU generators are not seeded per call: `simulate_for_sbi` seeds them once before it samples, so noise drawn on a GPU is reproducible with one worker only. `simulate_for_sbi` also seeds the sampling of the parameters. After a seeded call, the CPU random states are seeded with another seed derived from `seed`: the code that follows is reproducible, and later draws do not repeat those of the call.

Without a seed, one worker uses the current random state, and several workers seed every call from numpy's global random state.

For each call, `sbi` seeds the global random states of numpy, of torch on the CPU, and of Python's `random` module. Draw your noise from these, e.g. with `np.random.randn` or `torch.randn`. A numpy `Generator` that you create outside the simulator is not seeded; with several workers, every call even gets a copy of it in the same state, so the calls repeat each other's noise. Create it inside the simulator instead: `rng = np.random.default_rng(np.random.randint(2**32 - 1, dtype=np.int64))`.

Seeding needs joblib's default workers, which are separate processes. If you configure joblib to use threads, they share one random state, and a `seed` raises an error.

In [6]:
x1 = simulate_from_theta(
    simulator, theta, num_workers=4, seed=1, show_progress_bar=False
)
x2 = simulate_from_theta(
    simulator, theta, num_workers=2, seed=1, show_progress_bar=False
)
print("same seed, same simulations:", np.array_equal(x1, x2))

same seed, same simulations: True


## Troubleshooting

- **Pickling errors:** see the FAQ entry [When using multiple workers, I get a pickling error](../faq/question_03_pickling_error.md).
- **The program hangs, or simulations start over and over:** if your simulator lives in a module, do not start simulations when that module is imported. The workers import the module, which can deadlock or start the simulations again.
- **Every simulation is slow to start:** the simulator is sent to the workers together with the module-level objects it reads, unless the workers can import it. A simulator defined in a script or notebook is always sent. If it reads large objects, such as a lookup table, define it in a module file.